In [35]:
import tensorflow as tf
import tqdm
import torch
import torch.nn as nn
import numpy as np

In [2]:
from gpt_download import download_and_load_gpt2

In [3]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/checkpoint


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/encoder.json


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

File already exists and is up-to-date: gpt2/124M/model.ckpt.meta


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/vocab.bpe


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001


In [83]:
class transformer_block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x
class multi_head_attention(torch.nn.Module):
    def __init__(self, d_in,d_out,context_length,drop_out,num_heads,qkv_bias=False):
        super().__init__()
        
        assert (d_out%num_heads==0), \
            "d_out must be divisible by number of heads"
            
        self.d_out=d_out
        self.d_in=d_in
        self.head_dim=d_out//num_heads
        self.num_heads=num_heads
        self.out_proj=torch.nn.Linear(d_out, d_out)
        
        self.W_query=torch.nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key=torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.dropout=torch.nn.Dropout(drop_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length),diagonal=1) )
        
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        keys=self.W_key(x)
        values=self.W_value(x)
        query=self.W_query(x)
        
        # Now we split the b,num_tokens,d_out to b, num_tokens, num_heads, head_dim
        
        keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values=values.view(b, num_tokens, self.num_heads, self.head_dim)
        query=query.view(b, num_tokens, self.num_heads, self.head_dim)
        
        # Now group the matrices by head
        
        keys=keys.transpose(1,2) # b, num_heads, num_tokens, head_dim
        values=values.transpose(1,2)
        query=query.transpose(1,2)
        
        attention_scores = query @ keys.transpose(2,3)
        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(mask_bool, float('-inf'))
        
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        attention_weights=self.dropout(attention_weights) # shape is b, num_head,num_tokens,num_tokens
        context_vector =(attention_weights @ values).transpose(1,2)
        context_vector=context_vector.contiguous().view(b,num_tokens,self.d_out)
        context_vector= self.out_proj(context_vector)
        return context_vector
        
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.eps=eps
        self.scale=nn.Parameter(torch.ones(normalized_shape))
        self.shift=nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # This layer does nothing and just returns its input.
        mean=torch.mean(x,dim=-1,keepdim=True)
        var=torch.var(x,dim=-1,unbiased=False,keepdim=True)
        x=(x-mean)/torch.sqrt(var+self.eps)
        x=self.scale*x+self.shift
        return x

class GeluActivation(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self,x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2/torch.pi))*(x+0.044715*torch.pow(x,3))))

class feedforward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GeluActivation(),
            nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"]),
            
        )
        
    def forward(self,x):
      return self.layers(x)
  
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.attention=multi_head_attention(
                                       
                                             d_in=cfg["emb_dim"],
                                             d_out=cfg["emb_dim"],
                                             context_length=cfg["context_length"],
                                             num_heads=cfg["n_heads"],
                                             drop_out=cfg["drop_rate"],
                                             qkv_bias=cfg["qkv_bias"])
        self.layer_norm1=LayerNorm(normalized_shape=cfg["emb_dim"])
        self.layer_norm2=LayerNorm(normalized_shape=cfg["emb_dim"])
        self.ffn=feedforward(cfg)
        self.drop_shorcut=nn.Dropout(cfg["drop_rate"])
    
        
    def forward(self,x):
        shortcut=x
        x=self.layer_norm1(x)
        x=self.attention(x)
        x=self.drop_shorcut(x)
    
        x=x+shortcut
        shortcut=x
        x=self.layer_norm2(x)
        x=self.ffn(x)
        x=self.drop_shorcut(x)
        x=x+shortcut
        
        return x
                
class GPTModule(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.token_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        self.pos_dim=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
        self.drop=nn.Dropout(cfg["drop_rate"])
        self.trf_blocks=nn.Sequential(
            *[TransformerBlock(cfg) for _ in range (cfg["n_layers"])]
        )
        self.final_norm=LayerNorm(cfg["emb_dim"])
        self.out_head=nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self,in_idx):
        batch_size,seq_len=in_idx.shape
        token_emb=self.token_emb(in_idx)
        pos_emb=self.pos_dim(torch.arange(seq_len, device=in_idx.device))
        x=token_emb+pos_emb
        
        x=self.drop(x)
        x=self.trf_blocks(x)
        x=self.final_norm(x)
        logits=self.out_head(x)
        return logits


In [ ]:
print(f"setting: {settings}")
print(f"params: {params}")

In [85]:

params["wte"].shape #token embeddings

(50257, 768)

In [86]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [87]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])


In [88]:
NEW_CONFIG.update({"context_length":1024,
                  "qkv_bias":True})
gpt = GPTModule(NEW_CONFIG)
gpt.eval();

In [89]:
def assign(left,right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    else:
        return torch.nn.Parameter(torch.tensor(right))

In [90]:
def load_weight_into_gpt(params,gpt):
    gpt.pos_dim.weight=assign(gpt.pos_dim.weight,params["wpe"])
    gpt.token_emb.weight=assign(gpt.token_emb.weight,params["wte"])
    
    for i in range(len(params["blocks"])):
        q_w,k_w,v_w=np.split(
            params["blocks"][i]["attn"]["c_attn"]["w"],3,axis=-1
        ) 
        gpt.trf_blocks[i].attention.W_key.weight=assign(gpt.trf_blocks[i].attention.W_key.weight,k_w.T)
        gpt.trf_blocks[i].attention.W_query.weight=assign(gpt.trf_blocks[i].attention.W_query.weight,q_w.T)
        gpt.trf_blocks[i].attention.W_value.weight=assign(gpt.trf_blocks[i].attention.W_value.weight,v_w.T)
        
        q_b,k_b,v_b=np.split(
            params["blocks"][i]["attn"]["c_attn"]["b"],3,axis=-1
        )
        
        gpt.trf_blocks[i].attention.W_key.bias=assign(gpt.trf_blocks[i].attention.W_key.bias,k_b)
        gpt.trf_blocks[i].attention.W_value.bias=assign(gpt.trf_blocks[i].attention.W_key.bias,v_b)
        gpt.trf_blocks[i].attention.W_query.bias=assign(gpt.trf_blocks[i].attention.W_key.bias,q_b)
        
        gpt.trf_blocks[i].attention.out_proj.weight=assign(gpt.trf_blocks[i].attention.out_proj.weight, params["blocks"][i]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[i].attention.out_proj.bias=assign(gpt.trf_blocks[i].attention.out_proj.bias, params["blocks"][i]["attn"]["c_proj"]["b"])
        
        gpt.trf_blocks[i].ffn.layers[0].weight=assign(gpt.trf_blocks[i].ffn.layers[0].weight,params["blocks"][i]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[i].ffn.layers[0].bias=assign(gpt.trf_blocks[i].ffn.layers[0].bias,params["blocks"][i]["mlp"]["c_fc"]["b"])
        
        gpt.trf_blocks[i].ffn.layers[2].weight=assign(gpt.trf_blocks[i].ffn.layers[2].weight,params["blocks"][i]["mlp"]["c_proj"]["w"].T)   
        gpt.trf_blocks[i].ffn.layers[2].bias=assign(gpt.trf_blocks[i].ffn.layers[2].bias,params["blocks"][i]["mlp"]["c_proj"]["b"])
        
        
        gpt.trf_blocks[i].layer_norm1.scale = assign(
            gpt.trf_blocks[i].layer_norm1.scale, 
            params["blocks"][i]["ln_1"]["g"])
        gpt.trf_blocks[i].layer_norm1.shift = assign(
            gpt.trf_blocks[i].layer_norm1.shift, 
            params["blocks"][i]["ln_1"]["b"])
        gpt.trf_blocks[i].layer_norm2.scale = assign(
            gpt.trf_blocks[i].layer_norm2.scale, 
            params["blocks"][i]["ln_2"]["g"])
        gpt.trf_blocks[i].layer_norm2.shift = assign(
            gpt.trf_blocks[i].layer_norm2.shift, 
            params["blocks"][i]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])




In [91]:
device=torch.device("mps" if torch.mps.is_available() else "cpu")

In [92]:
load_weight_into_gpt(params,gpt)
gpt.to(device);

In [100]:
def generate(model,idx,max_new_tokens,context_size,temperature=1,top_k=None,eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond=idx[:, -context_size:]
        with torch.no_grad():
            logits=model(idx_cond)
        logits=logits[:,-1,:]
        
        if top_k is not None:
            top_logits,_=torch.topk(logits,20)
            min_val=top_logits[:,-1]
            logits=torch.where(logits<min_val, torch.tensor(float('-inf')).to(logits.device),logits)
        
        if temperature>0:
            logits =logits/temperature
            prob=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(prob,num_samples=1)
        else:
            prob=torch.softmax(logits,dim=-1)
            idx_next=torch.argmax(prob,dim=-1,keepdim=True)
        
        if idx_next==eos_id:
            break
        idx=torch.cat((idx,idx_next),dim=1)
        
    return idx

            
        
            

In [101]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))
tokenizer = tiktoken.get_encoding("gpt2")
start_context = "My name"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

tiktoken version: 0.12.0
encoded: [3666, 1438]
encoded_tensor.shape: torch.Size([1, 2])


In [104]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you toward more efficient and efficient processes, so in this situation, the more effort you put towards those activities, the longer you are


In [106]:
import urllib.request
from pathlib import Path
import os
import zipfile
import ssl

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

In [112]:
def download_and_unzip(url,zip_path,extracted_path,data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exist skipping download")
        return
    else:
        ssl_context=ssl._create_unverified_context()
        with urllib.request.urlopen(url, context=ssl_context) as response:
            with open(zip_path,"wb") as f:
                f.write(response.read())
                
        with zipfile.ZipFile(zip_path,'r') as f:
            f.extractall(extracted_path)
            
            # Add .tsv file extension
        original_file_path = Path(extracted_path) / "SMSSpamCollection"
        os.rename(original_file_path, data_file_path)
        print(f"File downloaded and saved as {data_file_path}")

download_and_unzip(url, zip_path, extracted_path, data_file_path)

                
            
        

File downloaded and saved as sms_spam_collection/SMSSpamCollection.tsv


In [113]:
import pandas as pd

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [114]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [120]:
def create_balanced_dataset(df):
    num_spam=df[df["Label"]=="spam"].shape[0]
    ham_df=df[df["Label"]=="ham"].sample(num_spam,random_state=123)
    
    balanced_df=pd.concat([df[df["Label"]=="spam"],ham_df])
    return balanced_df

balanced_dataset=create_balanced_dataset(df)

In [121]:
balanced_dataset["Label"]=balanced_dataset["Label"].map({"ham":0,"spam":1})

In [162]:
def random_split(df,train_frac,test_frac):
    
    df=df.sample(frac=1,random_state=123).reset_index(drop=True)
    
    train_size=int(len(df)*train_frac)
    test_size=int(len(df)*test_frac)

    
    train_df=df.iloc[:train_size,:]
    test_df=df.iloc[train_size:train_size+test_size,:]
    val_df=df.iloc[train_size+test_size:,:]
    
    return train_df,test_df,val_df

train_df, validation_df, test_df = random_split(balanced_dataset, 0.7, 0.1)

In [163]:
print(len(train_df))
print(len(validation_df))
print(len(test_df))

1045
149
300


In [164]:

data_=Path("Data_Processed")
data_.mkdir(exist_ok=True)
train_df.to_csv(os.path.join(data_,"train.csv"), index=None)
validation_df.to_csv(os.path.join(data_,"validation.csv"), index=None)
test_df.to_csv(os.path.join(data_,"test.csv"), index=None)

In [165]:
import torch
from torch.utils.data import Dataset

In [166]:
class spamDataset(Dataset):
    def __init__ (self,csv_file,tokenizer,max_len=None,pad_token=50256):
        self.df=pd.read_csv(csv_file)
        
        self.encoded_text=[tokenizer.encode(text) for text in self.df["Text"]]
        
        if max_len is None:
            self.max_len=self._max_len()
        
        else:
            self.max_len=max_len
            
            self.encoded_text=[encoded_text[:self.max_len] for encoded_text in self.encoded_text]
        
        self.encoded_text=[encoded_text+(self.max_len-len(encoded_text))*[pad_token] for encoded_text in self.encoded_text]
        
    def __getitem__(self, index):
        encoded=self.encoded_text[index]
        label=self.df.iloc[index]["Label"]
        
        return (
            torch.tensor(encoded,dtype=torch.long),
            torch.tensor(label,dtype=torch.long)
            
        )
        
    def __len__(self):
        return len(self.df)
        
    def _max_len(self):
        max_len=0
        for encoded_text in self.encoded_text:
            if len(encoded_text) > max_len:
                max_len=len(encoded_text)
            
        return max_len
    

In [167]:
train_dataset = spamDataset(
    csv_file=os.path.join(data_,"train.csv"),
    max_len=None,
    tokenizer=tokenizer
)
test_dataset = spamDataset(
    csv_file=os.path.join(data_,"test.csv"),
    max_len=None,
    tokenizer=tokenizer
    
)

val_dataset= spamDataset(
    csv_file=os.path.join(data_,"validation.csv"),
    max_len=None,
    tokenizer=tokenizer
)

print(train_dataset._max_len())

120


In [168]:
from torch.utils.data import DataLoader

In [169]:
batch_size=8
num_workers=1
train_data=DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
    
)

test_data=DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
    
)

val_data=DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)


In [170]:
print(f"{len(train_data)} training batches")
print(f"{len(val_data)} validation batches")
print(f"{len(test_data)} test batches")

130 training batches
19 validation batches
38 test batches


In [182]:
Choose_Model="gpt2-small (124M)"

BASE_CONFIG={
    "vocab_size":50257,
    "context_length":1024,
    "qkv_bias":True,
    "drop_rate": 0.0, 
}

model_configs={
        "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},

}

BASE_CONFIG.update(model_configs[Choose_Model])

assert train_dataset.max_len <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)


In [185]:
model_size=Choose_Model.split(" ")[-1].lstrip("(").rstrip(")")
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModule(BASE_CONFIG)
load_weight_into_gpt( params,model)
model.eval();

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/checkpoint


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/encoder.json


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

File already exists and is up-to-date: gpt2/124M/model.ckpt.meta


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/vocab.bpe


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001


In [190]:
text_1 = "Every effort moves you"

token_ids = generate_simpel_text(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_token=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [191]:
for params in model.parameters():
    params.requires_grad=False

In [198]:
torch.manual_seed(123)
num_classes=2
model.out_head=torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"],out_features=num_classes)


In [200]:
for params in model.trf_blocks[-1].parameters():
    params.requires_grad=True
    
for params in model.final_norm.parameters():
    params.requires_grad=True
    

In [203]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

Inputs: tensor([[5211,  345,  423,  640]])
Inputs dimensions: torch.Size([1, 4])


In [205]:
model.eval()
with torch.no_grad():
    outputs = model(inputs)
print(outputs)

tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])


In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("Class label:", label.item())

Class label: 1


In [ ]:
def calc_accuracy_loader(data_loader,device,model,num_batches=None):
    model.eval()
    correct_predictions, total_predictions=0,0
    if num_batches is None:
        num_batches=len(data_loader)
    else:
        num_batches=min(num_batches, len(data_loader))
        
    for i ,(input_batch,target_batch) in enumerate(data_loader):
        if i<num_batches:
            input_batch=input_batch.to(device)
            target_batch=target_batch.to(device)
            with torch.no_grad():
                logits=model(input_batch)[:,-1,:]
            predicted_label=torch.argmax(logits, dim=-1)
            total_predictions += predicted_label.shape[0]
            correct_predictions += (predicted_label==target_batch).sum().item()
        
        else:
            break
    return correct_predictions/total_predictions
            
            
                
    

In [210]:

device=torch.device("mps" if torch.mps.is_available() else "cpu")
model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes

torch.manual_seed(123) # For reproducibility due to the shuffling in the training data loader

train_accuracy = calc_accuracy_loader(train_data, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_data, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_data, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")

AttributeError: 'torch.device' object has no attribute 'eval'